# Capítulo 8 – Métricas e Avaliação

> Este notebook computa AUC, EER, calibração, WER/CER e métricas de cluster.

In [ ]:
# (Opcional) Instalar dependências em um ambiente local
# !pip install matplotlib scikit-learn evaluate

## 8.1 AUC e EER

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import numpy as np

np.random.seed(0)
y_true = np.r_[np.zeros(200), np.ones(200)]
y_score = np.r_[np.random.uniform(0,0.4,200), np.random.uniform(0.6,1.0,200)]

fpr, tpr, th = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)
eer = fpr[np.nanargmin(np.absolute((1 - tpr) - fpr))]
print(f"AUC: {auc:.4f}, EER: {eer:.4f}")

## 8.2 Calibração (Curva + Brier)

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt

prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=10)
print("Brier:", brier_score_loss(y_true, y_score))

plt.figure(figsize=(5,4))
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Probabilidade prevista"); plt.ylabel("Frequência observada")
plt.title("Curva de Calibração"); plt.tight_layout(); plt.show()

## 8.3 Métricas de clusterização

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=300, centers=3, n_features=10, random_state=42)
print("Silhouette:", silhouette_score(X, y))
print("Davies–Bouldin:", davies_bouldin_score(X, y))
print("Calinski–Harabasz:", calinski_harabasz_score(X, y))

## 8.4 WER/CER (exemplo simples)

In [ ]:
# Exemplo didático sem dependências externas
def cer(ref, hyp):
    # distância de Levenshtein em caracteres
    R, H = ref, hyp
    dp = [[i+j if i*j==0 else 0 for j in range(len(H)+1)] for i in range(len(R)+1)]
    for i in range(1,len(R)+1):
        for j in range(1,len(H)+1):
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1] + (R[i-1]!=H[j-1]))
    return dp[-1][-1]/len(R) if R else 0.0

def wer(ref, hyp):
    Rw, Hw = ref.split(), hyp.split()
    # distância de Levenshtein em palavras
    dp = [[i+j if i*j==0 else 0 for j in range(len(Hw)+1)] for i in range(len(Rw)+1)]
    for i in range(1,len(Rw)+1):
        for j in range(1,len(Hw)+1):
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1] + (Rw[i-1]!=Hw[j-1]))
    return dp[-1][-1]/len(Rw) if Rw else 0.0

ref = "ola mundo"
hyp = "ola mudo"
print("CER:", cer(ref, hyp))
print("WER:", wer(ref, hyp))